In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('weatherAUS.csv')

In [3]:
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


Target : Rain Tomorrow 

Problem Type: Classification

In [4]:
df['RainTomorrow'].value_counts(normalize=True)

RainTomorrow
No     0.775819
Yes    0.224181
Name: proportion, dtype: float64

This is an Imbalanced datset. 

Even if the model always predicts 'NO', it will get 77% accuracy. 

Hence we cannot evaluate model based on accuracy alone. We will use f1 score and recall also. 

In [5]:
df.shape

(145460, 23)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null

In [7]:
df.dtypes.value_counts()

float64    16
object      7
Name: count, dtype: int64

Observations: 

Rows: 145460

Total columns: 23

Numerical Features: 16

Categorical Features: 7

### Checking for the missing values 

In [8]:
df.isnull().sum()

Date                 0
Location             0
MinTemp           1485
MaxTemp           1261
Rainfall          3261
Evaporation      62790
Sunshine         69835
WindGustDir      10326
WindGustSpeed    10263
WindDir9am       10566
WindDir3pm        4228
WindSpeed9am      1767
WindSpeed3pm      3062
Humidity9am       2654
Humidity3pm       4507
Pressure9am      15065
Pressure3pm      15028
Cloud9am         55888
Cloud3pm         59358
Temp9am           1767
Temp3pm           3609
RainToday         3261
RainTomorrow      3267
dtype: int64

In [9]:
# There are multiple missing values. To get the correct idea, we need it in percentage form. 

In [10]:
df_null= pd.DataFrame(df.isnull().sum(), columns=['No of missing values'])

In [11]:
df_null['Percentage Missing']= df_null['No of missing values']/len(df)*100

In [12]:
df_null.sort_values(by="Percentage Missing", ascending=False)

,No of missing values,Percentage Missing
Sunshine,69835,48.009762
Evaporation,62790,43.166506
Cloud3pm,59358,40.807095
Cloud9am,55888,38.421559
Pressure9am,15065,10.356799
Pressure3pm,15028,10.331363
WindDir9am,10566,7.263853
WindGustDir,10326,7.098859
WindGustSpeed,10263,7.055548
Humidity3pm,4507,3.098446


Observation: 

1. The feature having approx more than 30% missing values can be dropped
2. Target column "Rain Tomorrow" is having missing values which cannot be allowed. So we will drop the rows which are having missing values. 
3. We will impute the missing values of other features 



In [13]:
cols_to_drop= df_null[df_null['Percentage Missing']>30].index
cols_to_drop

Index(['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm'], dtype='object')

In [14]:
#Dropping columns having more than 30% missing values
df=df.drop(cols_to_drop, axis=1)

In [15]:
# Dropping the rows which are having missing values
df=df.dropna(subset=['RainTomorrow'])

In [16]:
df.shape

(142193, 19)

In [17]:
df=df.copy()

### Date Feature 

- The date feature is object type. We have to convert it to datetime

In [21]:
df["Date"]=pd.to_datetime(df['Date'],  errors="coerce")

In [22]:
df["Date"].dtype


dtype('<M8[ns]')

In [23]:
df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month
df['DayOfWeek']=df['Date'].dt.dayofweek

In [26]:
df=df.drop(columns=['Date'])

### Converting Binary Columns
- Rain Tomorrow 
- Rain Today

In [29]:
df['RainTomorrow']=df['RainTomorrow'].map({'Yes':1, 'No': 0})
df['RainToday']=df['RainToday'].map({'Yes':1, 'No': 0})

In [31]:
df[['RainTomorrow', 'RainToday']].head()

,RainTomorrow,RainToday
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,0,0.0


In [32]:
X=df.drop("RainTomorrow", axis=1)
y=df["RainTomorrow"]

In [33]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=y)

In [34]:
X_train.shape

(113754, 21)

In [35]:
X_test.shape

(28439, 21)

In [40]:
num_cols=X_train.select_dtypes(include=['number']).columns
cat_cols=X_train.select_dtypes(include=['object']).columns

In [41]:
num_cols

Index(['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am',
       'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am',
       'Pressure3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'Year', 'Month',
       'DayOfWeek'],
      dtype='object')

In [42]:
cat_cols

Index(['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm'], dtype='object')

## Preprocessing Pipeline

In [44]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
num_pipeline=Pipeline(
    [("imputer", SimpleImputer(strategy='mean')),
     ("scaler", StandardScaler())
     
     ]
)


cat_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor= ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [50]:
from sklearn.ensemble import RandomForestClassifier

model_pipeline= Pipeline ([
    ("preprocessor",preprocessor),
    ("model", RandomForestClassifier(n_estimators=200,random_state=42,class_weight="balanced"))
])

In [51]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [52]:
y_pred=model_pipeline.predict(X_test)

In [53]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

[[21224   840]
 [ 3346  3029]]
              precision    recall  f1-score   support

           0       0.86      0.96      0.91     22064
           1       0.78      0.48      0.59      6375

    accuracy                           0.85     28439
   macro avg       0.82      0.72      0.75     28439
weighted avg       0.85      0.85      0.84     28439

ROC-AUC: 0.7185330944560565


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

models ={
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=0),
    "GradientBoosting": GradientBoostingClassifier()
}


In [57]:
for name, model in models.items():
    pipeline=Pipeline([
        ("preprocessor",preprocessor),
        ("model", model)
    ])

    pipeline.fit (X_train, y_train)
    y_pred=pipeline.predict(X_test)

    print(f"\n{name}")
    print(classification_report(y_test, y_pred))

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



LogisticRegression
              precision    recall  f1-score   support

           0       0.86      0.94      0.90     22064
           1       0.70      0.48      0.57      6375

    accuracy                           0.84     28439
   macro avg       0.78      0.71      0.74     28439
weighted avg       0.83      0.84      0.83     28439


RandomForest
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     22064
           1       0.77      0.48      0.59      6375

    accuracy                           0.85     28439
   macro avg       0.82      0.72      0.75     28439
weighted avg       0.84      0.85      0.84     28439


GradientBoosting
              precision    recall  f1-score   support

           0       0.86      0.95      0.91     22064
           1       0.74      0.48      0.58      6375

    accuracy                           0.85     28439
   macro avg       0.80      0.72      0.74     28439
weighted avg       0.8

Observation: Random Forest has given best performance

In [58]:
import joblib
joblib.dump(model_pipeline, "rainfall_rf_pipeline.pkl")

['rainfall_rf_pipeline.pkl']

### inference testing

In [59]:
pipeline=joblib.load("rainfall_rf_pipeline.pkl")

In [60]:
new_data = pd.DataFrame([{
    "Location": "Albury",
    "MinTemp": 12.5,
    "MaxTemp": 25.0,
    "Rainfall": 0.0,
    "WindGustDir": "W",
    "WindGustSpeed": 35.0,
    "WindDir9am": "W",
    "WindDir3pm": "NW",
    "WindSpeed9am": 15.0,
    "WindSpeed3pm": 20.0,
    "Humidity9am": 60.0,
    "Humidity3pm": 40.0,
    "Pressure9am": 1012.0,
    "Pressure3pm": 1010.0,
    "Temp9am": 18.0,
    "Temp3pm": 24.0,
    "RainToday": 0,
    "Year": 2016,
    "Month": 6,
    "DayOfWeek": 2
}])


In [61]:
prediction = pipeline.predict(new_data)
probability = pipeline.predict_proba(new_data)

print("Prediction:", prediction)
print("Rain Probability:", probability)

Prediction: [0]
Rain Probability: [[0.895 0.105]]
